# 03 — Representation Features: D-02 Surface Layer

이 노트북은 SSOT D-02 surface representation feature만 다룬다. 형태소(NB04), tokenizer/BPE mechanism(NB05/NB06), semantic embedding, QC, 모델링은 범위 밖이다.

## 0 Contract / Scope

candidate cohort(`pair_quality_status='accepted' AND analysis_eligible_exact_dedup AND logical_corpus IN ('025','026')`, N=3,836,013)에 대한 population 실행은 이 배정에서 명시적으로 승인됐다. Human N=500 audit 진행 여부와 무관하게 실행하며, 이 artifact는 `PROVISIONAL_ENGINEERING_OUTPUT`이지 formal G2 evidence가 아니다. 이 cohort는 Human audit 완료 전까지 CANDIDATE cohort다. token count/TP/logTP, Kiwi 형태소, regex chunk, BPE feature, semantic embedding, 새 QC/exclusion, 예측/모델링은 이 단계에서 금지한다.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from tokenization_premium.paths import PROJECT_ROOT
from tokenization_premium.representation import (
    COHORT_FILTER_SQL,
    REPRESENTATION_CONFIG_SHA256,
    REP_FEATURES_RELATIVE_PATH,
    compute_cohort_fingerprint,
    execute_representation_population,
)

FULL_RUN_AUTHORIZED = True
RESEARCH_SEMANTICS_FROZEN = True
FORBIDDEN_COMPONENTS = {
    'kiwi', 'morphology', 'o200k_token_count', 'tp', 'logtp', 'regex_chunk',
    'bpe', 'embedding_similarity', 'modeling', 'new_qc_exclusion',
}
assert FULL_RUN_AUTHORIZED and RESEARCH_SEMANTICS_FROZEN

PAIR_REGISTRY_V002 = PROJECT_ROOT / 'data/registry/PAIR_REGISTRY_v002.parquet'
REP_FEATURES_V001 = PROJECT_ROOT / REP_FEATURES_RELATIVE_PATH
FINGERPRINT_PATH = PROJECT_ROOT / 'outputs/manifests/CANDIDATE_COHORT_FINGERPRINT_v001.json'
MANIFEST_PATH = PROJECT_ROOT / 'outputs/manifests/REP_FEATURES_MANIFEST_v001.json'
{'cohort_filter_sql': COHORT_FILTER_SQL, 'config_sha256': REPRESENTATION_CONFIG_SHA256}

{'cohort_filter_sql': "pair_quality_status = 'accepted' AND analysis_eligible_exact_dedup AND logical_corpus IN ('025', '026')",
 'config_sha256': '75dcecb455443d79fc0509f5ab1f80ffe2b8f6364eeb04b426288fda7e6c8b67'}

## 1 Candidate cohort fingerprint verification

NB03 실행 전 생성된 fingerprint의 input SHA가 현재 `PAIR_REGISTRY_v002.parquet`과 여전히 일치하는지 확인한다. 새 dataset version을 만들지 않고, 기존 fingerprint를 그대로 재사용한다.

In [2]:
from tokenization_premium.hashing import sha256_file

fingerprint = json.loads(FINGERPRINT_PATH.read_text(encoding='utf-8'))
current_input_sha = sha256_file(PAIR_REGISTRY_V002)
assert fingerprint['input']['sha256'] == current_input_sha
assert fingerprint['filter_expression'] == COHORT_FILTER_SQL
{
    'n_pairs': fingerprint['n_pairs'],
    'sorted_pair_id_set_sha256': fingerprint['sorted_pair_id_set_sha256'],
    'source_counts': fingerprint['source_counts'],
    'direction_counts': fingerprint['direction_counts'],
    'domain_counts': fingerprint['domain_counts'],
}

{'n_pairs': 3836013,
 'sorted_pair_id_set_sha256': 'db4b2d66357fca6e6bad1dc611648aed5f4849dba6a6bba909d251aac14bd807',
 'source_counts': {'025': 2485974, '026': 1350039},
 'direction_counts': {'EN_TO_KO': 1273294,
  'KO_TO_EN': 2512172,
  'UNKNOWN': 50547},
 'domain_counts': {'dialogue': 516165,
  'general': 804296,
  'other': 2155645,
  'technology': 359907}}

## 2 Bounded pilot

N≈2,000 pilot으로 schema/null/range invariant를 먼저 검증한다. 문제가 없으면 full run으로 진행한다.

In [3]:
PILOT_OUTPUT = PROJECT_ROOT / '.runtime/nb03-pilot/REP_FEATURES_PILOT_v001.parquet'
PILOT_RUNTIME = PROJECT_ROOT / '.runtime/nb03-pilot'
if PILOT_OUTPUT.exists():
    PILOT_OUTPUT.unlink()

import datetime as dt
from zoneinfo import ZoneInfo
pilot_run_id = 'NB03_PILOT_' + dt.datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%dT%H%M%S')
pilot_manifest = execute_representation_population(
    project_root=PROJECT_ROOT,
    input_path=PAIR_REGISTRY_V002,
    output_path=PILOT_OUTPUT,
    runtime_dir=PILOT_RUNTIME,
    run_id=pilot_run_id,
    limit=2000,
)
{'run_mode': pilot_manifest['run_mode'], 'row_count': pilot_manifest['output']['row_count'], 'validation_status': pilot_manifest['validation_status']}

NB03:REPRESENTATION_TRANSFORM_WRITE:   0%|          | 0/2000 [00:00<?, ?it/s]

[10:44:36 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
0/2000
0.00%
throughput=NA
ETA NA
elapsed 00:00:00
RSS 1.91GiB
mem_available 10.35GiB
memory_status=OK
checkpoint=NONE


[10:44:36 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
2000/2000
100.00%
throughput=4831.57 items/s
ETA 00:00:00
elapsed 00:00:00
RSS 0.23GiB
mem_available 11.93GiB
memory_status=OK
checkpoint=REP_FEATURES_WRITTEN
[10:44:36 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
2000/2000
100.00%
throughput=4725.51 items/s
ETA 00:00:00
elapsed 00:00:00
RSS 0.23GiB
mem_available 11.93GiB
memory_status=OK
checkpoint=REP_FEATURES_WRITTEN


{'run_mode': 'PILOT', 'row_count': 2000, 'validation_status': 'PASS'}

In [4]:
import duckdb

con = duckdb.connect()
pilot_rel = f"read_parquet('{PILOT_OUTPUT.as_posix()}')"
checks = con.execute(f'''
    SELECT
        count(*) AS n,
        count(DISTINCT pair_id) AS distinct_pair_id,
        min(ko_hangul_share) AS min_ko_hangul_share,
        max(ko_hangul_share) AS max_ko_hangul_share,
        min(en_latin_share) AS min_en_latin_share,
        max(en_latin_share) AS max_en_latin_share,
        sum(CASE WHEN ko_codepoint_count <= 0 OR en_codepoint_count <= 0 THEN 1 ELSE 0 END) AS impossible_codepoints,
        sum(CASE WHEN ko_grapheme_count <= 0 OR en_grapheme_count <= 0 THEN 1 ELSE 0 END) AS impossible_graphemes
    FROM {pilot_rel}
''').fetchone()
con.close()
assert checks[0] == 2000
assert checks[1] == 2000
assert 0.0 <= checks[2] and checks[3] <= 1.0
assert 0.0 <= checks[4] and checks[5] <= 1.0
assert checks[6] == 0
assert checks[7] == 0
{'n': checks[0], 'distinct_pair_id': checks[1], 'impossible_codepoints': checks[6], 'impossible_graphemes': checks[7]}

{'n': 2000,
 'distinct_pair_id': 2000,
 'impossible_codepoints': 0,
 'impossible_graphemes': 0}

## 3 Full population run — PROVISIONAL_ENGINEERING_OUTPUT

Pilot 검증 통과 후 candidate cohort 전체(N=3,836,013)에 대해 full run을 진행한다. Human audit 진행 중에도 실행이 승인됐으며, 이 artifact는 formal G2 evidence로 주장하지 않는다.

In [5]:
if REP_FEATURES_V001.exists():
    raise FileExistsError('REP_FEATURES_v001.parquet이 이미 존재한다; 자동 재시작은 금지된다')

full_run_id = 'NB03_FULL_' + dt.datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%dT%H%M%S')
full_manifest = execute_representation_population(
    project_root=PROJECT_ROOT,
    input_path=PAIR_REGISTRY_V002,
    output_path=REP_FEATURES_V001,
    runtime_dir=PROJECT_ROOT / '.runtime/nb03-full',
    run_id=full_run_id,
)
{'run_mode': full_manifest['run_mode'], 'row_count': full_manifest['output']['row_count'], 'sha256': full_manifest['output']['sha256'], 'validation_status': full_manifest['validation_status']}

NB03:REPRESENTATION_TRANSFORM_WRITE:   0%|          | 0/3836013 [00:00<?, ?it/s]

[10:44:38 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
0/3836013
0.00%
throughput=NA
ETA NA
elapsed 00:00:00
RSS 2.80GiB
mem_available 9.45GiB
memory_status=OK
checkpoint=NONE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[10:44:48 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
100000/3836013
2.61%
throughput=9978.12 items/s
ETA 00:06:14
elapsed 00:00:10
RSS 2.66GiB
mem_available 9.51GiB
memory_status=OK
checkpoint=NONE


[10:44:58 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
220000/3836013
5.74%
throughput=10982.22 items/s
ETA 00:05:29
elapsed 00:00:20
RSS 2.66GiB
mem_available 9.65GiB
memory_status=OK
checkpoint=NONE


...

[10:50:05 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
3500000/3836013
91.24%
throughput=11654.97 items/s
ETA 00:00:28
elapsed 00:05:00
RSS 2.66GiB
mem_available 9.68GiB
memory_status=OK
checkpoint=NONE


[10:50:15 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
3600000/3836013
93.85%
throughput=11595.90 items/s
ETA 00:00:20
elapsed 00:05:10
RSS 2.66GiB
mem_available 9.71GiB
memory_status=OK
checkpoint=NONE


[10:50:28 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
3720000/3836013
96.98%
throughput=11608.13 items/s
ETA 00:00:09
elapsed 00:05:20
RSS 2.65GiB
mem_available 9.71GiB
memory_status=OK
checkpoint=NONE


[10:50:38 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
3836013/3836013
100.00%
throughput=11626.56 items/s
ETA 00:00:00
elapsed 00:05:29
RSS 0.37GiB
mem_available 10.99GiB
memory_status=OK
checkpoint=REP_FEATURES_WRITTEN
[10:50:38 KST]
phase=NB03
stage=REPRESENTATION_TRANSFORM_WRITE
3836013/3836013
100.00%
throughput=11626.26 items/s
ETA 00:00:00
elapsed 00:05:29
RSS 0.37GiB
mem_available 10.99GiB
memory_status=OK
checkpoint=REP_FEATURES_WRITTEN


{'run_mode': 'FULL_POPULATION',
 'row_count': 3836013,
 'sha256': '350172afd05a13e13ee7dc24d61e3af6bc6a9215bf685d2b3ade8e946e6975b4',
 'validation_status': 'PASS'}

## 4 Full artifact validation

row count, pair_id 유일성, cohort fingerprint와의 정합성을 최종 확인한다.

In [6]:
con = duckdb.connect()
full_rel = f"read_parquet('{REP_FEATURES_V001.as_posix()}')"
row_count, distinct_pair_id = con.execute(f'SELECT count(*), count(DISTINCT pair_id) FROM {full_rel}').fetchone()
fingerprint_match = con.execute(f'''
    SELECT count(*) FROM {full_rel} r
    JOIN read_parquet('{PAIR_REGISTRY_V002.as_posix()}') p USING (pair_id)
    WHERE {COHORT_FILTER_SQL}
''').fetchone()[0]
con.close()
assert row_count == fingerprint['n_pairs']
assert distinct_pair_id == fingerprint['n_pairs']
assert fingerprint_match == fingerprint['n_pairs']
{'row_count': row_count, 'distinct_pair_id': distinct_pair_id, 'fingerprint_join_match': fingerprint_match}

{'row_count': 3836013,
 'distinct_pair_id': 3836013,
 'fingerprint_join_match': 3836013}

## 5 Manifest persistence

실행 요약과 cohort fingerprint 참조를 `REP_FEATURES_MANIFEST_v001.json`으로 저장한다.

In [7]:
full_manifest_out = dict(full_manifest)
full_manifest_out['cohort_fingerprint'] = {
    'sorted_pair_id_set_sha256': fingerprint['sorted_pair_id_set_sha256'],
    'n_pairs': fingerprint['n_pairs'],
}
MANIFEST_PATH.write_text(json.dumps(full_manifest_out, ensure_ascii=False, indent=2), encoding='utf-8')
{'manifest_path': str(MANIFEST_PATH.relative_to(PROJECT_ROOT)), 'status': full_manifest_out['status']}

{'manifest_path': 'outputs/manifests/REP_FEATURES_MANIFEST_v001.json',
 'status': 'PROVISIONAL_ENGINEERING_OUTPUT'}